In [1]:
import torch
import torch.nn as nn

In [2]:
batch_size = 64
sequence_len = 64
embedding_size = 32
head_size = 256
steps = 5000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len)

In [4]:
# model declaration
class RecurrentNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size)
        self.input_linear = nn.Linear(in_features=embedding_size, out_features=head_size)
        self.hidden_linear = nn.Linear(in_features=head_size, out_features=head_size)
        self.output_linear = nn.Linear(in_features=head_size, out_features=vocab_size)        

    def forward(self, input, h=None):
        batch_size = input.size(dim=0)
        if h is None:
            h = torch.zeros(size=(batch_size, head_size))
        logits = []
        for i in range(input.size(dim=1)):
            ch = input[:, i]
            new_h = torch.tanh(input=self.input_linear(input=self.embedding(input=ch)) + self.hidden_linear(input=h))
            logits.append(self.output_linear(input=new_h))
            h = new_h
        return torch.stack(tensors=logits, dim=1), h

In [5]:
# model initialization
model = RecurrentNeuralNetwork()
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

In [6]:
# training
for step in range(steps):
    optimizer.zero_grad()
    
    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    train_logits, _ = model(input=x_train[batch])
    train_loss = nn.functional.cross_entropy(input=train_logits.view(-1, vocab_size), target=y_train[batch].view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits, _ = model(input=x_val)
            val_loss = nn.functional.cross_entropy(input=val_logits.view(-1, vocab_size), target=y_val.view(-1))
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.211015701293945, val loss = 4.208025932312012


Step 1000: train loss = 1.5522186756134033, val loss = 1.711281657218933


Step 2000: train loss = 1.5244922637939453, val loss = 1.6644448041915894


Step 3000: train loss = 1.4485218524932861, val loss = 1.645324945449829


Step 4000: train loss = 1.4040236473083496, val loss = 1.6438456773757935


Step 4999: train loss = 1.4619295597076416, val loss = 1.630388855934143


In [7]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000
    h = None

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)            

    while len(output) < max_output_length:
        logits, h = model(input=i, h=h)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[output[-1]]]], dtype=torch.long)

In [8]:
print(''.join(output))

ROMEO:
And pleast that know know thoe hanged no morrow,
Unlessance of accersude of 'What: upuntips of second.

CLARENCE:
What vail wleady?

Gid me, I'll not and defend, and hast soul? But that prisoner,
With tonatural true troy
Hath death father, this wouldst biles of Clarght for his puss'd ourselves
Of alg our party,
Show them newly me.

LOHERSOTIS XFan Hent nigh to hang for our prispent myself a it my help on,
I crue woest to duke, do but not told the grace, yet ignorant, it desier:
I play, thou die.

ROPSOD:
If it sigh; you must she well do you, myself noe is now: for truth in anceed Some graving took I be hour,
Unfow: being dieg:
Hers you.

KING HENRY VI:
It is lose the
pitch wouse'er mercy.
What I did give me,
To do, Marcius, to red Lothingward gick.

ROMETH:
O heavens?

TYRREL:
Ament.

LADY CAPULET:
What is thy slowers damnngs he wousin,
Misusher,'st Tybalt against this knee: I loved thy possessions if known minest thou Edward fall of vile, steal holbscale, of it. Tranish men,
Wu